#  Credit Risk Analysis — EDA & Model Comparison

**Objective**: Explore the German Credit Dataset, build and rigorously compare credit scoring models, and interpret results using industry-standard metrics.

This notebook covers 5 parts:
1. Data Loading & Exploratory Data Analysis (EDA)
2. Feature Engineering
3. Model Comparison (Logistic Regression, Random Forest, Gradient Boosting, XGBoost)
4. Evaluation (AUC-ROC, Gini Coefficient, KS Statistic, Lift Curve)
5. Interpretability (SHAP)

> **Dataset**: [German Credit Data](https://archive.ics.uci.edu/ml/datasets/statlog+(german+credit+data)) — 1,000 loan applications, 20 features, target `credit_risk` (1 = good, 0 = default).


## 0. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
)
from sklearn.calibration import calibration_curve

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available — will be skipped")

import shap
shap.initjs()

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RANDOM_STATE = 42
print("✅ All imports OK")


---
## 1. Data Loading & Exploratory Data Analysis

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/selva86/datasets/master/GermanCredit.csv"
df = pd.read_csv(DATASET_URL)

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Target: credit_risk  (1 = good payer, 0 = default)")
df.head()


In [ ]:
df.describe(include="all").T.round(2)


In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "✅ No missing values")


### 1.1 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df["credit_risk"].value_counts()
labels = ["Good payer (1)", "Default (0)"]
colors = ["#4CAF50", "#F44336"]

axes[0].bar(labels, counts.values, color=colors, width=0.5, edgecolor="white")
axes[0].set_title("Observation count")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=labels, colors=colors,
            autopct="%1.1f%%", startangle=90,
            wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Class distribution (%)")

fig.suptitle("Class Imbalance — German Credit Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n⚠️  Imbalance ratio: {counts[1]/counts[0]:.1f}:1 (good:default)")
print("→ Requires class_weight='balanced' or resampling strategy")


### 1.2 Numerical Features

In [ ]:
num_cols = ["duration", "amount", "installment_rate", "age", "number_credits", "people_liable"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for label, color, name in [(0, "#F44336", "Default"), (1, "#4CAF50", "Good payer")]:
        subset = df[df["credit_risk"] == label][col]
        axes[i].hist(subset, bins=25, alpha=0.6, color=color,
                     label=name, edgecolor="white", density=True)
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)

plt.suptitle("Numerical Feature Distributions by Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
corr = df[num_cols + ["credit_risk"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title("Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 1.3 Categorical Features

In [ ]:
cat_cols = ["credit_history", "purpose", "employment_duration", "savings"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ct = df.groupby([col, "credit_risk"]).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="barh", ax=axes[i], color=["#F44336", "#4CAF50"],
                edgecolor="white", width=0.7)
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].set_xlabel("% within category")
    axes[i].legend(["Default", "Good payer"], loc="lower right", fontsize=8)
    axes[i].xaxis.set_major_formatter(mtick.PercentFormatter())

plt.suptitle("Default Rate by Category", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 2. Feature Engineering

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    employment_map = {"unemployed": 0, "< 1 yr": 0.5,
                      "1 <= ... < 4 yrs": 2.5, "4 <= ... < 7 yrs": 5.5, ">= 7 yrs": 8.0}
    out["employment_length"] = out["employment_duration"].map(employment_map).fillna(2.0)

    # Debt-to-income proxy
    income_proxy = (out["installment_rate"] * 1100) + (out["employment_length"] * 700)
    out["dti_ratio"] = out["amount"] / income_proxy.clip(lower=1)
    out["dti_ratio"] = out["dti_ratio"].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Credit history length proxy
    out["credit_history_length"] = (out["age"] - 18).clip(lower=0) + 0.25 * out["duration"]

    # Past delinquencies flag
    ch_text = out["credit_history"].str.lower()
    out["has_delinquency"] = ch_text.str.contains("delay|critical|overdue", regex=True).astype(int)

    # Interaction: high amount × long duration = higher risk
    out["amount_x_duration"] = out["amount"] * out["duration"] / 1e6

    return out

df_fe = feature_engineering(df)
new_features = ["employment_length", "dti_ratio", "credit_history_length",
                "has_delinquency", "amount_x_duration"]
print("Engineered features summary:")
print(df_fe[new_features].describe().round(3))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ["dti_ratio", "credit_history_length", "amount_x_duration"]):
    for label, color, name in [(0, "#F44336", "Default"), (1, "#4CAF50", "Good payer")]:
        subset = df_fe[df_fe["credit_risk"] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True, edgecolor="white")
    ax.set_title(col.replace("_", " ").title())
    ax.legend(fontsize=9)

plt.suptitle("Engineered Feature Distributions by Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 3. Data Preparation & Pipelines

In [ ]:
NUMERIC_FEATURES = [
    "duration", "amount", "age", "installment_rate",
    "number_credits", "people_liable",
    "dti_ratio", "credit_history_length", "has_delinquency",
    "employment_length", "amount_x_duration"
]
CATEGORICAL_FEATURES = [
    "credit_history", "purpose", "employment_duration",
    "savings", "status", "housing", "job"
]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "credit_risk"

X = df_fe[ALL_FEATURES].copy()
y = df_fe[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Default rate — Train: {1-y_train.mean():.1%} | Test: {1-y_test.mean():.1%}")

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), CATEGORICAL_FEATURES),
])


---
## 4. Model Comparison

### 4.1 Model Definitions

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("clf", LogisticRegression(
            max_iter=2500, class_weight="balanced",
            solver="liblinear", C=0.1, random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=200, max_depth=8,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("prep", preprocessor),
        ("clf", GradientBoostingClassifier(
            n_estimators=200, learning_rate=0.05,
            max_depth=4, subsample=0.8, random_state=RANDOM_STATE
        ))
    ]),
}

if HAS_XGB:
    scale_pos_weight = (y_train == 1).sum() / (y_train == 0).sum()
    models["XGBoost"] = Pipeline([
        ("prep", preprocessor),
        ("clf", XGBClassifier(
            n_estimators=200, learning_rate=0.05, max_depth=4,
            subsample=0.8, scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE, eval_metric="auc",
            verbosity=0, use_label_encoder=False
        ))
    ])

print(f"{len(models)} models defined: {list(models.keys())}")


### 4.2 Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
print("5-fold stratified cross-validation...\n")

for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train,
                             cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:25s}  AUC = {scores.mean():.4f} ± {scores.std():.4f}")

print("\n✅ Cross-validation complete")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds  = [cv_results[n].std()  for n in names]
colors_cv = ["#5C6BC0", "#43A047", "#FB8C00", "#E53935"][:len(names)]

bars = ax.barh(names, means, xerr=stds, color=colors_cv,
               alpha=0.85, edgecolor="white", height=0.55,
               error_kw=dict(elinewidth=1.5, capsize=4, ecolor="gray"))

for bar, mean in zip(bars, means):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f"{mean:.4f}", va="center", fontsize=10, fontweight="bold")

ax.set_xlabel("AUC-ROC (5-fold CV)")
ax.set_title("Model Comparison — Cross-Validation AUC-ROC", fontsize=13, fontweight="bold")
ax.set_xlim(0.5, 1.0)
ax.axvline(0.7, color="red", linestyle="--", alpha=0.5, label="Acceptable threshold (0.70)")
ax.legend()
plt.tight_layout()
plt.show()


### 4.3 Test Set Evaluation

In [ ]:
trained_models = {}
test_results = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred  = pipeline.predict(X_test)

    auc  = roc_auc_score(y_test, y_proba)
    gini = 2 * auc - 1
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    ks_stat = np.max(tpr - fpr)

    test_results[name] = {
        "y_proba": y_proba, "y_pred": y_pred,
        "auc": auc, "gini": gini, "ks": ks_stat,
        "fpr": fpr, "tpr": tpr
    }

summary = pd.DataFrame({
    name: {
        "AUC-ROC":         f"{r['auc']:.4f}",
        "Gini Coefficient": f"{r['gini']:.4f}",
        "KS Statistic":    f"{r['ks']:.4f}",
    }
    for name, r in test_results.items()
}).T

print("Test set metrics:\n")
print(summary.to_string())
summary


### 4.4 ROC Curves

In [ ]:
colors_roc = ["#5C6BC0", "#43A047", "#FB8C00", "#E53935"]
fig, ax = plt.subplots(figsize=(8, 6))

for (name, r), color in zip(test_results.items(), colors_roc):
    ax.plot(r["fpr"], r["tpr"], color=color, lw=2,
            label=f"{name} (AUC={r['auc']:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier (AUC=0.500)")
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR)")
ax.set_title("ROC Curves — Model Comparison", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


### 4.5 Cumulative Lift Curve

> The lift curve is an industry standard in credit scoring: it shows how many defaults you capture by targeting the top X% riskiest applicants.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for (name, r), color in zip(test_results.items(), colors_roc):
    idx = np.argsort(r["y_proba"])[::-1]
    y_sorted = np.array(y_test)[idx]
    x_pct = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
    cumdefault = np.cumsum(1 - y_sorted) / (1 - y_test).sum()
    ax.plot(x_pct * 100, cumdefault * 100, color=color, lw=2, label=name)

perf_x = [(1 - y_test).mean() * 100, 100]
ax.plot([0] + perf_x, [0, 100, 100], "g--", lw=1.5, label="Perfect model")
ax.plot([0, 100], [0, 100], "k--", lw=1.2, label="Random baseline")

ax.set_xlabel("% of population targeted (sorted by descending score)")
ax.set_ylabel("% of defaults captured")
ax.set_title("Cumulative Lift Curve (Gain Chart)", fontsize=13, fontweight="bold")
ax.legend()
ax.set_xlim(0, 100); ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

print("Top-30% capture rate per model:")
for name, r in test_results.items():
    idx = np.argsort(r["y_proba"])[::-1]
    y_s = np.array(y_test)[idx]
    n30 = int(0.3 * len(y_s))
    capture = (1 - y_s[:n30]).sum() / (1 - y_test).sum()
    print(f"  {name:25s}: {capture:.1%} of defaults in top 30%")


### 4.6 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(test_results), figsize=(4*len(test_results), 4))
if len(test_results) == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, test_results.items()):
    cm = confusion_matrix(y_test, r["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Default (0)", "Good (1)"],
                yticklabels=["Default (0)", "Good (1)"],
                ax=ax, cbar=False, linewidths=0.5)
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")

plt.suptitle("Confusion Matrices (test set)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
## 5. Probability Calibration

> In credit scoring, predicted **probabilities must be calibrated** — a predicted PD of 0.3 should correspond to a ~30% actual default rate. This is required for regulatory use (Basel II/III IRB approach).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for (name, r), color in zip(test_results.items(), colors_roc):
    fraction_pos, mean_pred = calibration_curve(y_test, r["y_proba"], n_bins=10)
    ax.plot(mean_pred, fraction_pos, "s-", color=color, lw=2, ms=6, label=name)

ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of actual positives")
ax.set_title("Calibration Curve (Reliability Diagram)", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print("Note: Logistic Regression is typically better calibrated than tree ensembles.")
print("For production use, apply Platt Scaling or Isotonic Regression to ensemble models.")


---
## 6. Interpretability — SHAP Values

> SHAP (SHapley Additive exPlanations) explains **why** the model assigns a given score to each applicant. Required in regulated environments (Basel III, GDPR right to explanation).

In [ ]:
lr_pipeline = trained_models["Logistic Regression"]
X_test_transformed = lr_pipeline.named_steps["prep"].transform(X_test)
feature_names = lr_pipeline.named_steps["prep"].get_feature_names_out()

explainer = shap.LinearExplainer(
    lr_pipeline.named_steps["clf"],
    X_test_transformed,
    feature_perturbation="correlation_dependent"
)
shap_values = explainer.shap_values(X_test_transformed)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Number of features after encoding: {len(feature_names)}")


In [ ]:
shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=feature_names,
    max_display=15, show=False
)
plt.title("SHAP — Global Feature Impact (Logistic Regression)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
shap.summary_plot(
    shap_values, X_test_transformed,
    feature_names=feature_names,
    plot_type="bar", max_display=12, show=False
)
plt.title("SHAP — Top 12 Features by Mean Absolute Impact", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
high_risk_idx = np.argmax(lr_pipeline.predict_proba(X_test)[:, 0])

print(f"Sample applicant (index {high_risk_idx}):")
print(f"  Predicted default probability: {1 - lr_pipeline.predict_proba(X_test)[high_risk_idx, 1]:.1%}")
print(f"  Actual class: {'Default' if y_test.iloc[high_risk_idx] == 0 else 'Good payer'}")

shap.force_plot(
    explainer.expected_value,
    shap_values[high_risk_idx],
    feature_names=feature_names,
    matplotlib=True, show=False,
    figsize=(18, 3)
)
plt.title(f"Individual Explanation — High-Risk Applicant", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 7. Decision Threshold Optimization

> In credit scoring, **rejecting a good applicant** (false positive) and **approving a defaulter** (false negative) have asymmetric costs. We can optimize the decision threshold using a cost matrix.

In [ ]:
COST_FN = 5   # Cost of false negative (approving a defaulter)
COST_FP = 1   # Cost of false positive (rejecting a good applicant)

best_model_name = max(test_results, key=lambda n: test_results[n]["auc"])
r = test_results[best_model_name]

thresholds = np.linspace(0.01, 0.99, 200)
costs = []
for t in thresholds:
    y_pred_t = (r["y_proba"] < t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    if cm.shape == (2, 2):
        fn = cm[0, 1]
        fp = cm[1, 0]
        costs.append(COST_FN * fn + COST_FP * fp)
    else:
        costs.append(np.nan)

optimal_threshold = thresholds[np.nanargmin(costs)]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, costs, color="#5C6BC0", lw=2)
ax.axvline(optimal_threshold, color="red", linestyle="--", lw=1.5,
           label=f"Optimal threshold = {optimal_threshold:.2f}")
ax.axvline(0.5, color="gray", linestyle=":", lw=1.5, label="Default threshold = 0.50")
ax.set_xlabel("Decision threshold")
ax.set_ylabel(f"Total cost (FN×{COST_FN} + FP×{COST_FP})")
ax.set_title(f"Threshold Optimization — {best_model_name}", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nOptimal threshold: {optimal_threshold:.3f} (vs 0.5 default)")
print(f"Minimum cost: {min(costs):.0f}")


---
## 8. Summary & Results

In [ ]:
print("=" * 65)
print(f"{'Model':<25} {'AUC-ROC':>8} {'Gini':>8} {'KS Stat':>8}")
print("=" * 65)
for name, r in sorted(test_results.items(), key=lambda x: -x[1]["auc"]):
    print(f"{name:<25} {r['auc']:>8.4f} {r['gini']:>8.4f} {r['ks']:>8.4f}")
print("=" * 65)

winner = max(test_results, key=lambda n: test_results[n]["auc"])
print(f"\nBest model (AUC): {winner}")
print(f"  AUC  = {test_results[winner]['auc']:.4f}")
print(f"  Gini = {test_results[winner]['gini']:.4f}")
print(f"  KS   = {test_results[winner]['ks']:.4f}")


 

### References
- Baesens et al. (2003) — *Benchmarking State-of-the-Art Classification Algorithms for Credit Scoring* — Journal of the Operational Research Society
- Hand & Henley (1997) — *Statistical Classification Methods in Consumer Credit Scoring* — Journal of the Royal Statistical Society
- Basel II/III — Internal Ratings-Based (IRB) Approach for PD estimation
